# Single-Agent vs Multi-Agent

**Prerequisites:** from `02_agent_runtime`: `05_sub_agents.ipynb`

Chapter 2 already runs several agents at once. A `Think` calls `delegate`, the work splits into branches, and each branch runs its own loop with its own clean context. So before building anything, this notebook has to answer a plain question: **is that already a multi-agent system?**

The answer it arrives at is no. A branch is the same agent again with a narrower goal: the same system prompt, the same persona, the same tools. A multi-agent system is several agents that are *different*, because somebody designed each of them for its own job.

**By the end you'll have:**
- Seen, without calling a model, that a chapter 2 branch starts with its parent's system prompt, and that `delegate` has no field that could give it a different one
- Written an agent as a *definition*: a name, a persona, and the tools it may use
- Given one job that needs five different people to three setups: one agent, a chapter 2 fork, and five specialists
- Read what each produced, piece by piece, and measured what each cost
- A rule for when to use one agent, when a fork is ideal, and when the job needs specialists

## Setup

This notebook runs on **`openai/gpt-oss-20b`**, the smaller of the two gpt-oss models Groq serves. It is cheaper, it has its own daily quota, and a smaller model shows the strain of doing five jobs at once sooner than a bigger one does.

One adjustment is needed first. gpt-oss-20b stops at 2,048 output tokens unless told otherwise, and that count includes its hidden reasoning. On a long task it can spend the whole allowance thinking and return an empty answer. `Think` has no setting for this, so a small subclass adds one:

## A branch is a copy

When `Act` starts a branch, it builds the branch's conversation in `_seed_branch`. Chapter 2 chose what goes in on purpose: not the outer history, which made a branch answer other branches' questions, but *system messages plus the goal*.

That was the right fix for the problem chapter 2 had. It also means the thing that makes an agent who it is, its system prompt, is copied from the parent. Nothing here calls a model:

The branch meant to write a formal legal notice starts with a system prompt describing *both* jobs, the warm one included.

The list of fields is the other half. Per branch, the delegating model can set a goal, whether it may fork again, whether to explain itself, sampling (`temperature`, `top_p`) and budgets. There is no field for a system prompt and none for tools. `branch_runtime(branch)` does receive this dict, so a factory *could* build a different runtime per branch, but the dict holds nothing to build a different agent *from*. So chapter 2's factory builds the same agent every time.

That leaves one way for a branch to become someone in particular: the parent writes it into the `goal`, on the spot, in its own words.

## An agent is a definition

Chapter 1's `Agent` already had the right shape: `Agent(name, system, tools, ...)`. What it lacked was chapter 2's loop. Putting the two together takes a few lines:

This is the chapter's idea in one class. A `persona` and a set of `tools` are fixed when the agent is *defined*, by you. A branch inherits both.

Nobody in this notebook needs a tool, so every agent here gets an empty `Registry`. The field is there because tools are half of what makes an agent different, and later notebooks use it.

## The job

A batch of headphones has a battery fault, and the company is recalling it. Answering that takes five pieces of writing, and in a real company five different people would write them. Each wants something different from the same facts:

| Who | Writes | For | What they care about |
|---|---|---|---|
| **Ana**, support lead | the email to customers | a worried owner | that they are safe and know exactly what to do |
| **Marcus**, PR director | the press statement | journalists | trust, proportion, nothing that invites a follow-up story |
| **Deborah**, product lawyer | the notice to the regulator | a regulator, and maybe a court | every material fact, and no admission of fault |
| **Raj**, lead engineer | the internal engineering brief | other engineers | the real numbers, the real uncertainty, the supplier named |
| **Zoe**, social media manager | the social post | someone scrolling | one action and one link, in two lines |

These pull against each other. Raj wants every number and Marcus wants as few as possible. Ana says sorry like a person and Deborah never says anything that sounds like fault. One brief of facts goes to everyone, including a section marked internal only.

## Three ways to do it

**One agent.** One writer, with one system prompt: *"You handle the company's written communications about a product recall"*, followed by instructions for each kind of piece and the brief. The instructions are the same five jobs the specialists do, written as rules for one writer rather than as five people, so nobody is named. It is asked for all five pieces in a single reply.

What goes where matters. The system prompt says *who the agent is*. The user message says *what to do now*, including how to lay out the answer. Only the system prompt ever reaches a branch.

**A chapter 2 fork.** The same system prompt as the one agent, one persona with the instructions for all five pieces and the brief, with a short request: *"Coordinate the company's written response to a product recall. Write the full response: the customer email, the press statement, the regulator notice, the internal engineering brief and the social post. Split the work between five sub-agents, one per piece."* Each branch is built the way chapter 2 builds branches: the parent's system prompt, plus the goal the parent writes.

**Five specialists.** Five `Agent`s, each with one persona plus the brief, each asked for its own piece. They run side by side and their pieces are simply collected. How a supervisor would decide who does what is notebook 2. Here the split is wired by hand, so that the one thing that differs between the three setups is *who* does the writing.

The fork stops after three steps: the parent decides to delegate, the branches run, and their results come back. Left to go on, the parent's next step would be to read all five pieces back and type them out again as its own answer. In an earlier run, that one request was bigger than the 8,000 tokens per minute this account is allowed on gpt-oss-20b, and Groq refused it outright. The pieces are already written by then, so nothing is lost by stopping.

## What each agent starts with

Every agent's first request is its system prompt plus its task, and every later call carries at least that much again. `Think._build_messages()` returns what would be sent without sending it, so it can be counted for free. A branch's task is the goal the parent writes, which does not exist until the fork runs, so its row shows only what it gets before that.

- **The one agent starts with 1,076 tokens:** one persona, the instructions for all five pieces, the brief, and the request.
- **The fork's parent starts with 1,640:** the same, plus the `delegate` schema and the guidance on when to use it.
- **A fork branch starts with 1,021 before its goal is added:** the parent's whole system prompt, all five sets of instructions included, to write *one* piece.
- **A specialist starts with about 620 to 660:** its own instructions, the brief, and its piece. Everything it carries is about the one piece it writes.

## Running all three

A `Meter` wraps the one client every component shares, so each run's calls and tokens come from the API's own `usage` field rather than an estimate. It also notes which task each call was working on, which is how the fork's branches get told apart afterwards.

### One agent

### A chapter 2 fork

### Five specialists

## Reading what came back

These are single runs, so read them as examples, not scores. Every setup makes ordinary slips: invented dates (2024 appears in the one agent's pieces and the fork's), an invented hotline in the fork's email, *"Forty‑zero units"* and an invented company name from the specialists. The brief's instruction not to invent anything cut these down but did not stop them.

**The one agent is the clear loser.** It wrote all five pieces in one reply of 1,003 output tokens, and the pieces bled into each other:

- The **press statement** tells journalists the company is *"working with our supplier, Voltcell, to resolve the issue."* The supplier is under INTERNAL ONLY in the brief, and its responsibility is in dispute.
- The **regulator notice** lists *"7. Supplier of affected cells: Voltcell, lot VC‑7731"* and *"9. Estimated recall cost: $410,000."* Both are internal.
- The **social post** ends *"No fire risk, just battery heat."* Nothing in the brief says that. It plays down a safety risk in the one piece most people will read.
- The **engineering brief** says *"Two customers with legal representation"*. It was one.

**The fork did about as well as the specialists.** Its regulator notice and press statement keep every internal fact out. Its engineering brief is blunt, with the supplier, the dispute and the cost where they belong. Its email leads with the action. One real weakness: the regulator notice leaves out the investigation (9 of 11 units with a separator defect, 2 undetermined, 400 tested clean). The instructions call that material, and a regulator would expect it.

**The specialists** wrote in voice, with the brief's facts, and kept internal facts out of the public pieces, as in their saved run above.

This was not the only run. Across every run made while building this notebook, the regulator notice carried an internal fact in **4 of 6** runs of the one agent, **2 of 6** of the fork (set up in slightly different ways over those runs), and **1 of 4** of the specialists. The numbers are small, but the one agent is consistently the worst, and the fork and the specialists are close.

**Why the fork recovered most of the quality:** each branch had one piece to write, in a fresh context. That is what the one agent lacked. It had to hold five audiences in one reply, and each piece got a fifth of its attention.

But the fresh context is only half of it. The other half is that the parent, on its own initiative, copied each piece's rules and facts into that branch's goal. The next section shows why that matters so much.

## What the branches were given

Every branch was started by `_seed_branch`: the parent's system prompt, then its goal. Here is what that system prompt held, and for each goal, whether it named the person writing and which of the brief's key facts it repeated:

Every branch's system prompt holds the parent's whole prompt: all five sets of instructions and the brief. No one is named in it, so there is no persona for a goal to point at. The parent wrote goals of 65 to 158 tokens, and each one restates the instructions for its piece and the facts it needs:

> *"Write a formal, exact regulator notice. Include every material fact: hazard, affected units, corrective action. Do not admit fault or liability…"*

So a branch is the parent copied, handed a smaller problem with a well-written brief for it.

**That is what kept the voices apart, and it was the parent's choice, not the fork's design.** Look at what a branch would have without it. Its system prompt holds the rules for all five pieces side by side: be warm, be formal, be blunt, never name the supplier, always name the supplier. A goal that only says *"write the regulator notice"* tells it which piece, but leaves it to sort out which of those rules are its own, with the other four sets right there in its prompt. The five voices share one prompt, and they mix.

An earlier version of this notebook shows exactly that. Its goals named the piece and little else, *"Write a formal regulator notice in the voice of DEBORAH…"*. The branch that wrote the regulator notice listed the supplier, the supplier's dispute, the cost and the engineer's private worry about a wider fault. That is the engineer's content, taken from the prompt the two branches shared.

So a fork's quality rests on the parent getting each goal right: the rules for that piece, and the facts it needs, written out at run time in the parent's own words. When it does, a branch behaves like a specialist. When it doesn't, the branch falls back on a system prompt that was written for all five pieces at once. Nothing in the fork checks which of those happened.

It can go further wrong still. A version of this notebook gave the parent a one-line system prompt and put the brief only in its request, so branches inherited no facts at all. That parent wrote one-line goals with none of the brief's facts in them, and its branches invented a different incident: a regulator notice about headphones *"igniting"*, a press statement saying *"No injuries have been reported"*, an engineering brief counting *"1,047,832 units"*.

A specialist does not depend on any of this. Its rules and facts were written once, by you, before the run, and it has no other piece's rules to confuse them with.


## What it cost

- **The one agent was cheapest by far:** one call, 2,156 tokens, 3 seconds.
- **The specialists cost 3.3× as much:** five copies of the brief, and a full piece from each.
- **The fork cost 6.5× as much, about twice the specialists.** Each branch starts with about 400 more tokens than a specialist, since it carries the instructions for all five pieces, and five branches make that about 2,000. The parent's call carries the `delegate` schema. Its goals are long, because it restated each piece's instructions and facts. And its branches wrote longer pieces: 6,367 output tokens against the specialists' 3,628. It was also stopped before its final answer, which would have been the biggest request of the run.

The seconds are noisy: Groq is fast, and both the fork and the specialists back off when five agents at once hit the 8,000 tokens-per-minute cap.

So the fork bought most of the specialists' quality, at about twice their price. It got that quality because the parent happened to copy each piece's rules into its goal. A parent that didn't would have paid the same and got mixed voices.

## Which one to use

| | Use it when | What it gives you | What it costs |
|---|---|---|---|
| **One agent** | the job is small, and every part of it wants the same person | the cheapest answer | every part shares one reply and one context, so parts get thin and bleed into each other |
| **Fork** | the job is **too big for one context**, but **every part wants the same person** | each part gets a fresh, short context and one problem to solve | the parent's system prompt in every branch; if the parts need different rules, the result depends on the parent writing each part's rules into its goal |
| **Specialists** | the parts of the job need **different people** | each agent carries only its own instructions, fixed in advance by you | you design and maintain each one |

**A fork is ideal when the job is bigger than one context should hold, but one persona can do every part of it.** Reviewing forty files against the same coding standard. Summarising ten long reports for the same reader. Checking each clause of a contract with the same lens. Researching five competitors the same way. Each branch is the same agent, copied, with one piece of the work and a context that holds only that piece. Nothing in the parent's system prompt is wrong for any branch, because every branch is meant to be the same person. That is where copying costs nothing you would want to change, and the fresh, short context is pure gain. In this notebook, that is what lifted the fork above the one agent.

**Specialists are right when the parts need different people:**

- **Different voices.** Warm, formal, blunt and casual in one job.
- **Goals that pull against each other.** The engineer wants every number; the press statement wants as few as possible.
- **Things one of them must say and another must not.** The supplier's name belongs in the engineering brief and nowhere else.
- **Different tools.** Nobody here needed one, but an agent that must not be able to send an email or move money is a reason to split on its own.

A fork can imitate specialists, as it did here, but only when the parent writes each branch a goal that restates that part's rules and facts. That is the parent improvising a specialist at run time, in its own words, every time, and nothing checks that it did. When it doesn't, every branch writes from one system prompt meant for all the parts, and the voices mix. Specialists fix each part's rules once, in advance, carry nothing else, and cost less per piece.

Some reasons that are not enough on their own:

- **Context isolation.** Forks and specialists both have it. It is a reason to split the work, not a reason to use different agents.
- **Parts that can run in parallel.** Branches already do that.

The specialists here were wired up by hand: five agents, each given its piece. Notebook 2 gives them a supervisor that reads the job and decides which agent gets what. A chapter 2 parent cannot do that, because it has no list of agents to choose from.